<a href="https://colab.research.google.com/github/laura-williams-1/Machine_Learning_project2/blob/perceptron/P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---

```
CISC 3440: Machine Learning
P2 (35 Points + 5 Extra-Credit Points)
Due: Nov 9th, 11:59 pm
```
---
**Submission Instructions**


* You are expected to complete this project on your own (not with a partner).
* Please refer to the syllabus for the late submission policy.
* You must submit your assignment via Brightspace.
* Your code should work without errors; code that fails will not be graded.


We will work with the Kaggle Dataset (https://www.kaggle.com/datasets/ivansher/nasa-nearest-earth-objects-1910-2024) for this project. The dataset involves predicting if a nearest object to earth is Hazardous.

The project will involve two parts:
1. In the first part you will implement the Perceptron Algorithm and Train the model using the data
2. In the second part you will use Linear Regression to build a model for the same


You are provided with `train.csv` and starter code. Split this file once into 90% training data and 10% validation data. Use Pandas to load and inspect the data, and use the Pandas and Regression tutorials as guides.

**A separate test set is withheld for final evaluation.** Each saved preprocessing pipeline and model will be evaluated on the raw test features. Do not use test data to fit preprocessing, balance classes, select an epoch, or choose model settings.

At the final step, save both models and all fitted preprocessing together in one file, `P2_model.pkl`. No separate preprocessing or model files are required.

#Part-1 Perceptron

For the first part you will implement the Perceptron Algorithm, which you will use to train the data. You can use the implementation of the algorithm provided in your slides, **but be prepared to explain the algorithm to me in person if asked**.

**[5 points] Task-1: Perceptron Algorithm**

Implement `perceptron_algorithm(training_samples, training_labels, iterations)`. Inputs are a feature array of shape `(N, D)`, labels of shape `(N,)` containing `-1` or `+1`, and the number of epochs. Here, `N` is the number of samples and `D` is the number of features. One epoch is one complete pass through the training data.

Return two lists, `weight_values` and `bias_values`, containing the weights and bias after each epoch. Each list must contain `iterations` entries. Store an independent copy of each weight vector, with shape `(D,)`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

def perceptron_algorithm(training_samples,training_labels,iterations):
  """
  Train a perceptron for the specified number of epochs.

  Parameters
  ----------
  training_samples : numpy.ndarray, shape (N, D)
      Feature array with N samples and D features.
  training_labels : numpy.ndarray, shape (N,)
      Labels containing -1 or +1, one per sample.
  iterations : int
      Number of epochs. Each epoch is one full pass through the training data.

  Returns
  -------
  weight_values : list of numpy.ndarray
      Independent copies of the weight vectors after each epoch, each with
      shape (D,). The list contains iterations entries.
  bias_values : list of float
      Bias values after each epoch. The list contains iterations entries.

  Notes
  -----
  Initialize the weights and bias randomly.

  Example
  -------
  >>> training_samples = np.array([[1, 2], [2, 3], [-1, -1]])
  >>> training_labels = np.array([1, 1, -1])
  >>> weight_values, bias_values = perceptron_algorithm(training_samples, training_labels, iterations=10)
  """




**[2 Points] Task-2: Predictions**

Implement `perceptron_predict(test_sample, w, b)` for one sample. Compute `w @ test_sample + b`. Return `+1` if the result is nonnegative; otherwise return `-1`.


In [ ]:
def perceptron_predict(test_sample,w,b):
  """
  Predict the class label for one sample.

    Parameters
    ----------
    test_sample : numpy.ndarray, shape (D,)
        Feature values for one sample.
    w : numpy.ndarray, shape (D,)
        Weight vector.
    b : float
        Bias value.

    Returns
    -------
    int
        +1 if w @ test_sample + b >= 0; otherwise -1.

    Example
    -------
    >>> test_sample = np.array([2, 3])
    >>> w = np.array([1, 0.5])
    >>> b = -1
    >>> perceptron_predict(test_sample, w, b)
    1
  """
  # create score variable for activation score
  score = 0
  # loop thru the len of test sample array
  for i in range(len(test_sample)):
    # compute activation score per each iteration, add to score
    score += test_sample[i] * w[i]
  # outside of loop add bias to score
  score += b
  # if score >= 0 predict 1
  if score >= 0:
    return 1
  # if score < 0 predict -1
  else:
    return -1

### Dataset Features

Each row describes a near-Earth object. Use these five numerical columns as model inputs:

| Feature | Meaning |
|---|---|
| `absolute_magnitude` | Brightness: lower values mean brighter objects. |
| `estimated_diameter_min` | Lower estimate of the object's diameter. |
| `estimated_diameter_max` | Upper estimate of the object's diameter. |
| `relative_velocity` | How fast the object moves relative to Earth. |
| `miss_distance` | How closely the object approaches Earth. |

The **target** is `is_hazardous`: `True` means potentially hazardous; `False` means non-hazardous. Do not include this column in the input features.


**[1 point] Task-3: Visualize feature and label distributions**

Load `train.csv` into a Pandas DataFrame named `trainData` and use `info()` to inspect its columns and missing values. Explore the original data before cleaning, scaling, or balancing it.

**You may search online or use LLMs to explore different ways to visualize the dataset.** Choose plots that help you understand the data, and be prepared to explain your code and what the plots show.

Keep these two analyses separate:

1. **Feature distributions:** Visualize each of the five numerical input features. Compare their ranges, scales, and shapes. What do you notice that could affect model training or motivate feature scaling?
2. **Label distribution:** Visualize the counts or proportions of `True` and `False` in `is_hazardous`. Are the classes balanced? What might this mean for training and evaluation?

Add your code in the empty cell below and briefly describe your observations in your written answers.


In [ ]:
# FILL IN: visualize the feature distributions and label distribution separately.
import pandas as pd


### Prepare raw features and split the data

The code below selects the five raw feature columns, encodes the labels as `-1` and `+1`, and creates a 90% training / 10% validation split. The fixed seed (`random_state=42`) makes the split reproducible, and `stratify=labels` approximately preserves the class proportions. Reuse this split throughout the project; preprocessing and balancing come afterward.


In [ ]:
from sklearn.model_selection import train_test_split

feature_columns = [
    "absolute_magnitude", "estimated_diameter_min", "estimated_diameter_max",
    "relative_velocity", "miss_distance",
]
features = trainData[feature_columns].copy()
labels = trainData["is_hazardous"].map({False: -1, True: 1}).astype(int)

train_features_raw, val_features_raw, train_labels_raw, val_labels_raw = train_test_split(
    features, labels, test_size=0.1, random_state=42, stratify=labels
)
print("Training samples:", len(train_features_raw))
print("Validation samples:", len(val_features_raw))


**[4 Points] Task-4: Cleaning and preprocessing**

Based on `trainData.info()`, answer:

1. Are all five feature columns numeric?
2. Which feature columns contain missing values?
3. Why do the Boolean labels need to be encoded as `-1` and `+1`?

Use the Regression tutorial to build a preprocessing `Pipeline` named `preprocessor`. It should handle missing feature values and, if you choose to rescale the features, apply your chosen scaler. You might find it useful to rescale your features; explain your choice using the feature distributions.

- **Fit only on training data:** use `preprocessor.fit_transform(train_features_raw)` for training inputs.
- **Transform validation data without refitting:** use `preprocessor.transform(val_features_raw)`.
- Keep the raw DataFrames unchanged for later experiments. The pipeline must accept the five raw feature columns and produce one finite numerical feature vector per input row, including when some input values are missing.
- Keep this fitted pipeline for saving with the perceptron model. Refer to the tutorial's Pipeline example; do not include the target column in the pipeline inputs.

Create these NumPy arrays:

| Variable | Shape |
|---|---|
| `train_features_numpy` | `(N, D)` |
| `train_labels_numpy` | `(N,)` |
| `val_features_numpy` | `(M, D)` |
| `val_labels_numpy` | `(M,)` |

Here, `N` and `M` are the training and validation sample counts, and `D` is the number of transformed features.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

preprocessor = None
train_features_numpy = None
val_features_numpy = None
train_labels_numpy = None
val_labels_numpy = None
# FILL IN: build the preprocessing pipeline, fit it on training data only,
# transform validation data, and convert the label Series to one-dimensional arrays.


**[2 points] Task-5: Training and validation**

Train for **50 epochs** (full passes through the training data). You may change `iterations`, but use the same value before and after balancing.

For each saved epoch, predict validation labels and use scikit-learn's `accuracy_score` and `f1_score` to compute:

- Accuracy → `accuracy_over_iterations`
- Compute F1 for the hazardous class (`+1`) and store one score per epoch in `f1_score_over_iterations`.

Store one real score per epoch in each list, plot both against epoch number, and retain the original-balance results for comparison with balanced training.


In [ ]:
iterations = 50  # You may change this value; use it for both perceptron experiments.
w_unbalanced, b_unbalanced = perceptron_algorithm(
    train_features_numpy, train_labels_numpy, iterations
)

from sklearn.metrics import accuracy_score, f1_score

accuracy_over_iterations = []
f1_score_over_iterations = []
# FILL IN: use w_unbalanced and b_unbalanced to compute one validation score per epoch.

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(accuracy_over_iterations) + 1), accuracy_over_iterations, label="Accuracy")
plt.plot(range(1, len(f1_score_over_iterations) + 1), f1_score_over_iterations, label="F1 (hazardous)")
plt.xlabel("Epoch")
plt.ylabel("Validation score")
plt.title("Perceptron: original training class balance")
plt.legend()
plt.show()


**[1 Point] Task-6: Interpret the validation results**

1. How do accuracy and F1-score vary across epochs?
2. If accuracy and F1 differ, how might the class distribution help explain the difference?


**[6 Points] Task-7: Balancing the training dataset**

Observe your earlier label-distribution plot. Identify the majority and minority classes, and explain how the imbalance could affect accuracy and F1-score.

Use [`RandomUnderSampler`](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.RandomUnderSampler.html) from `imblearn.under_sampling` with `random_state=42` to balance **only the training data**. Its `fit_resample` method accepts `train_features_raw` (a DataFrame) and `train_labels_raw` directly, and returns matching balanced features and labels. By default, it reduces the majority class to the minority-class size. If needed, install the package with `%pip install imbalanced-learn`.

In the code cell below:

1. Apply the sampler and combine its returned features and encoded labels into `df_balanced`, using `is_hazardous` as the label-column name. Keep each row paired with its label. Compare class counts before and after balancing.
2. Transform `df_balanced[feature_columns]` with the existing fitted `preprocessor`, **without refitting**. Store the transformed features in `balanced_train_features_numpy` and the encoded labels (`-1`, `+1`) as a one-dimensional NumPy array in `balanced_train_labels_numpy`.

Train again using the same `iterations` value and evaluate each epoch on the original `val_features_numpy` and `val_labels_numpy`. **Do not balance the validation set or create another split.**

Optionally, try oversampling (increasing the minority-class size) on the training partition and compare the results.


In [ ]:
from imblearn.under_sampling import RandomUnderSampler

df_balanced = None
balanced_train_features_numpy = None
balanced_train_labels_numpy = None
# FILL IN: use RandomUnderSampler to balance the raw training data.
# Create df_balanced with its encoded is_hazardous column and compare class counts.
# Use the fitted preprocessor to transform the balanced features without refitting,
# and convert the labels to a one-dimensional NumPy array.


In [ ]:
# Reuse iterations (50 by default). Change it above to change both experiments.
w, b = perceptron_algorithm(
    balanced_train_features_numpy, balanced_train_labels_numpy, iterations
)

balanced_accuracy_over_iterations = []
balanced_f1_score_over_iterations = []
# FILL IN: evaluate each entry in w and b on the unchanged validation arrays.

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(balanced_accuracy_over_iterations) + 1),
         balanced_accuracy_over_iterations, label="Accuracy")
plt.plot(range(1, len(balanced_f1_score_over_iterations) + 1),
         balanced_f1_score_over_iterations, label="F1 (hazardous)")
plt.xlabel("Epoch")
plt.ylabel("Validation score")
plt.title("Perceptron: balanced training data, original validation distribution")
plt.legend()
plt.show()


**[2 points] Task-8: Compare and select the perceptron**

How did balancing the training data change validation accuracy and F1-score? Compare the original-balance and balanced-training results.

Choose an epoch from the balanced-training experiment using validation F1-score, and explain your choice. `chosen_iteration` is a **zero-based list index**: index 0 means epoch 1.

Keep copies of the selected weights and bias in `selected_weights` and `selected_bias`. You will save them with both models' preprocessing and the regression model in the final step below.


In [ ]:
chosen_iteration = None  # FILL IN: zero-based index of the selected balanced-training epoch.
selected_weights = None
selected_bias = None
if chosen_iteration is not None:
    selected_weights = w[chosen_iteration].copy()
    selected_bias = float(b[chosen_iteration])


## Part II: Linear Regression

**[12 points] Task-9: Train, validate, and select a regression model**

Use the same raw balanced training rows in `df_balanced` and the same original validation rows from Part I. **Do not create another split or balance the validation set.**

Follow the Regression tutorial to explore these choices:

- Whether polynomial or other basis functions improve the model.
- Whether to use ordinary least squares or Ridge regression, and which Ridge `alpha` values to try.

Build a complete scikit-learn `Pipeline` named `regression_pipeline` that includes missing-value handling, any feature expansion and rescaling, and the regression estimator. Fit it on `df_balanced[feature_columns]` and its encoded labels only. Use `regression_pipeline.predict(val_features_raw)` to obtain validation scores; the pipeline applies its fitted transformations without refitting.

Convert each score to a class label: **`+1` if the score is greater than or equal to zero; otherwise `-1`.** Compute validation accuracy and binary F1 with hazardous objects as the positive class. Use validation results to choose your final model and explain your decisions.

Keep your selected complete fitted pipeline in `regression_pipeline`. The final saving cell stores it together with the perceptron and its fitted `preprocessor` in one file. Every fitted preprocessing step must use training data only.

The saved pipeline will be evaluated on the raw withheld test features, with its scores converted to labels using the same zero threshold. The withheld test set is used only for final evaluation, not model selection.


In [ ]:
regression_pipeline = None
# FILL IN: fit and compare complete regression pipelines using balanced training data
# and the unchanged validation set. Keep your selected fitted pipeline above.


## Part III: Optional extra credit

**[5 extra-credit points] Task-10: Improve the F1-score**

Can you improve on your earlier models? Try other classifiers covered in the course, such as a decision tree or K-nearest neighbors (KNN). You may use grid search and cross-validation to explore hyperparameters.

- Reuse the original `train_features_raw`, `train_labels_raw`, and unchanged validation split. Use the same five input features and encoded labels (`-1`, `+1`).
- For cross-validation, use stratified folds **within the training partition only**. Put preprocessing in the estimator pipeline so it is fitted separately within each fold. If you resample, do so only inside each fold's training data (for example, with an imbalanced-learn pipeline). Do not balance or fit preprocessing on validation folds.
- Report the algorithms, hyperparameter values, and search method you tried. If you use cross-validation, report the number of folds and its F1 results. Use fixed random seeds where supported. You may use a stratified subset of the training partition for an initial search; report its size and fit the selected model on the training partition afterward.
- Evaluate your selected model on the original validation set. Include a table comparing its hazardous-class F1 (`pos_label=1`) with your best original-balance perceptron, balanced-training perceptron, and regression model. Also report precision and recall and briefly explain the result. Select any probability threshold using training-only cross-validation.

**To earn the 5 extra-credit points, your optional model must achieve a validation F1-score strictly higher than every model in your earlier tasks, with a reproducible comparison that follows the rules above.** There is no required absolute F1 target. The withheld test set remains reserved for final evaluation.

Keep your selected model and all fitted preprocessing in `extra_credit_pipeline`. Calling `extra_credit_pipeline.predict(raw_features)` must return class labels (`-1` or `+1`); if you tune a threshold, include that prediction rule in the saved estimator. The final saving cell includes this optional pipeline in the same bundle. Leave it as `None` if you skip this task.


In [ ]:
extra_credit_pipeline = None
# OPTIONAL FILL IN: compare classifiers and hyperparameters using training data only.
# Keep the selected fitted pipeline above; predict(raw_features) must return -1 or +1.
# Report validation F1, precision, and recall alongside your earlier models.


### Final step: save the bundle and submit

Run the saving cell below after selecting both models. It creates **one file**, `P2_model.pkl`, containing:

- The input feature-column order, label mapping, and prediction threshold.
- The fitted `preprocessor` and selected perceptron weights and bias.
- The complete fitted `regression_pipeline`, including its preprocessing and regression model.
- The fitted `extra_credit_pipeline`, if you complete Task-10 (`None` otherwise).

No separate imputer, scaler, or model files are needed. Before submitting, reload this bundle and check that all submitted models reproduce their validation predictions from the raw validation features. Use the stored feature-column order and do not refit preprocessing.

This same bundle will be loaded to evaluate all submitted models on raw features from a withheld test set. Keep the selected models trained on the training partitions used above; do not refit them on validation data before saving.

Submit **three files** through Brightspace, replacing `(lastname)` with your last name:

1. `answers_(lastname).txt`: written answers and explanations of your choices.
2. `P2_(lastname).ipynb`: completed code, comments, plots, and validation results.
3. `P2_model_(lastname).pkl`: the single bundle containing the required models, any optional extra-credit model, and all fitted preprocessing.

The cell saves `P2_model.pkl`; add your last name to that filename before submission. Restart the kernel and run all cells in order before submitting.


In [ ]:
import pickle

if selected_weights is None or selected_bias is None or regression_pipeline is None:
    raise ValueError("Select both models before running the final saving cell.")

p2_bundle = {
    "feature_columns": feature_columns,
    "preprocessor": preprocessor,  # Fitted preprocessing used by the perceptron.
    "perceptron_weights": selected_weights.copy(),
    "perceptron_bias": float(selected_bias),
    "regression_pipeline": regression_pipeline,  # All regression preprocessing and the model.
    "extra_credit_pipeline": extra_credit_pipeline,  # Optional classifier; predict returns -1/+1.
    "label_mapping": {False: -1, True: 1},
    "threshold": 0.0,  # Perceptron and regression score threshold; optional classifier uses predict().
}
with open("P2_model.pkl", "wb") as file:
    pickle.dump(p2_bundle, file)
print("Saved the required models, optional extra-credit model, and fitted preprocessing to P2_model.pkl.")
